# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM-self-improving-ai-agents-course/blob/main/hands_on/session_02_HANDS_ON_reward_function.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# Timer

In [1]:
SET_TIMER = False  # False, True, or minutes as a number

import requests, types
url = "https://raw.githubusercontent.com/Nicolepcx/ORM-self-improving-ai-agents-course/main/timer.py"

timer = types.ModuleType("timer")
exec(requests.get(url).text, timer.__dict__)

timer.start_exam_timer(enabled=SET_TIMER, minutes=15, warn_minutes=5)

# About this Notebook

## Train a Small Model with RL-Style Feedback (ART + RULER)

Welcome to this hands-on lab. You will train a small open model to perform a **custom task** reinforcement learning.

This notebook is essentially: **RL intuition applied to instruction following.**

<br>

This lab demonstrates the core building blocks behind **self-improving agents**.


## 1. What you are actually training

You are not training a model from scratch. You start from a strong **base model**:

* `BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"` or similar.

Then you train a **LoRA adapter** on top, using [ART’s training loop](https://art.openpipe.ai/getting-started/about). That is why the model is lightweight to train and fast to iterate on.

### The Mapping

| Component | This Notebook | RL Framing |
| --- | --- | --- |
| **State** | The prompt content, including the system prompt and user input | Current observation |
| **Action** | The model’s next token at each decoding step | Action sequence |
| **Policy** | Transformer weights (base) + LoRA adapter (trainable) | Stochastic policy |
| **Trajectory** | Messages plus the assistant completion | Episode transcript |
| **Reward** | Judge score in `[0, 1]` | Scalar return |

---

## 2. The loop: generate, judge, learn

At each training step, the notebook does three things:

### A. Generate rollouts

For each training input, the model produces multiple candidate outputs:

* `rollouts_per_group = 2`

This is the core idea behind relative methods like GRPO or RULER-style learning: you do not need one perfect label, you need **comparisons** and **ranking signals**.

### B. Judge rollouts (RULER-style)

A separate judge model scores each candidate output:

* `RULER_MODEL = "openrouter/deepseek/deepseek-r1-0528"`

The judge is instructed to return strict JSON and provide a per-candidate score:

* `1.0` means the output matches the task format and intent
* `0.0` means it violates the format or ignores the task

This notebook uses `robust_score_group(...)` which is resilient to:
* code fences
* extra text around JSON
* partial or malformed responses

### C. Update the policy

ART then trains the LoRA adapter so that high-reward rollouts become more likely.

This is the same conceptual move as policy optimization in RL:
* good behavior becomes more probable
* bad behavior is discouraged

---

## 3. Task descriptions are your “reward specification”

The most important control knob in this lab is the task description:

```python
TASK_DESCRIPTION = GRAMMARLY_TASK_DESCRIPTION
````


Note: Parts of the Notebook are adapted from the [ART examples](https://art.openpipe.ai/getting-started/notebooks)


This notebook is for the *Hands-on* for Session 2 for Develop Self-Improving AI Agents with Reinforcement Learning Live Event with O'Reilly Media by
[Nicole Koenigstein](https://www.linkedin.com/in/nicole-koenigstein/).

<font color="red" size="5">
<b>Attention for the Notebook to work </b>
</font>
<br>

you need an `OPENROUTER_API_KEY`! [Get your key here](https://openrouter.ai/)   

# Installation

In [2]:
%%capture
!pip install -U --force-reinstall \
  "gql>=4.0.0,<5" \
  "weave>=0.52" \
  "openpipe-art" \
  tenacity \
  pillow==11.3.0 \
  protobuf==5.29.5 \
  --no-cache-dir

In [3]:
import gql
import gql.transport.exceptions as exc

print("gql version:", gql.__version__)
print("has TransportConnectionFailed:", hasattr(exc, "TransportConnectionFailed"))

from gql.transport.exceptions import TransportConnectionFailed
print("gql OK")

gql version: 4.0.0
has TransportConnectionFailed: True
gql OK


In [4]:
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Set API Keys

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# OpenRouter normal path
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Anthropic compatibility path redirected to OpenRouter
os.environ["ANTHROPIC_BASE_URL"] = "https://openrouter.ai/api"
os.environ["ANTHROPIC_AUTH_TOKEN"] = OPENROUTER_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ""  # must be explicitly empty


In [6]:
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required for data generation and RULER evaluation.")

# Optional W&B
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    print("WANDB_API_KEY is not set. We'll skip logging metrics to Weights & Biases.")


# Imports

In [7]:
import os
import re
import json
import random
from typing import List, Optional

import torch
import weave
from dotenv import load_dotenv
from litellm import acompletion
from pydantic import BaseModel, Field

import art
from art.local import LocalBackend
from art.utils import iterate_dataset
from art.utils.litellm import convert_litellm_choice_to_openai
import torch
from unsloth import FastLanguageModel

from google.colab import userdata
from huggingface_hub import login, whoami, create_repo


/tmp/ipykernel_659/323292841.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Settings

In [8]:
# Model configuration
MODEL_NAME = "jira-model-001"  # Name for your trained model
PROJECT_NAME = "auto-rl"  # Project name for tracking


# Training configuration
TRAINING_CONFIG = {
    "num_training_inputs": 25,  # Number of training inputs to generate
    "groups_per_step": 1,  # Inputs to process per training step
    "num_epochs": 3,  # Number of times through all data
    "rollouts_per_group": 2,  # Different responses per input (for RULER comparison)
    "learning_rate": 1e-5,  # Learning rate
    "max_training_steps": None,  # Maximum training steps (set to None for no limit)
}

NUM_TEST_INPUTS = 5  # Number of test inputs to generate
RULER_MODEL = "openrouter/deepseek/deepseek-r1-0528"  # Model for RULER evaluation
SYSTEM_PROMPT_GENERATION_MODEL = "openrouter/moonshotai/kimi-k2"
INPUT_GENERATION_MODEL = "openrouter/moonshotai/kimi-k2"


# GPU configuration (keep these as-is unless you have a reason to change them, since the setup already leverages almost all memor yfor a A100 with 40GB)
MAX_SEQ_LENGTH = 2048  # Maximum sequence length
GPU_MEMORY_UTILIZATION = 0.6  # GPU memory usage (0.0-1.0)


# Hands-on

<font color="red" size="10">
<b>TODO: </b>
</font>
<br>
<font color="black" size="5">
<b>Teach your model another skill, set <code>TASK_DESCRIPTION</code> to one of the descriptions from Sample Taks or create an own description.</b>
</font>



# Sample Tasks

In [9]:
GRAMMARLY_TASK_DESCRIPTION = """
Read the user's text and check if it has any grammar or spelling errors. If it does, then fix them by wrapping the
erroneous text in <original></original> tags and the corrected text in <corrected></corrected> tags.

For example, if the user's text is "I are going to the store to buy sum eggs", the output should be:

I <original>are</original><corrected>am</corrected> going to the store to buy <original>sum</original><corrected>some</corrected> eggs.
"""

PM_TO_CODER_TASK_DESCRIPTION = """
Convert the user's project manager style text into clear, actionable coding instructions.

Output format must be STRICT and follow exactly this schema, in this order:

TITLE: <one line>
GOAL: <one line>
CONTEXT: <one paragraph, optional if missing>
REQUIREMENTS:
- <bullet>
- <bullet>
ACCEPTANCE_CRITERIA:
- <bullet>
- <bullet>
EDGE_CASES:
- <bullet>
- <bullet>
QUESTIONS:
- <bullet questions that must be clarified, if any>

Rules:
- Preserve intent. Remove fluff, buzzwords, and vague phrases.
- If something is ambiguous, do not guess. Put it into QUESTIONS.
- If the user mentions a system, UI, API, database, auth, or performance, reflect that in REQUIREMENTS or EDGE_CASES.
- Keep it concise and engineering ready.
"""

EMOJIFY_TASK_DESCRIPTION = """
Convert any incoming story provided by the user into a corresponding sequence of emojis.
For example, if the user says, "I went to the store to buy some eggs but forgot my wallet",
you should convert it into something like:"🚶‍♂️➡️🏬🛒🥚…😱💳❌".
"""

CHANGELOG_TASK_DESCRIPTION = """
Convert the user's description into:
- a concise Git commit message (imperative mood)
- a short changelog entry for end users

Format:
COMMIT:
CHANGELOG:

Rules:
- Commit is max 72 characters.
- Changelog is 1 to 3 sentences, non technical.
"""

LINKEDIN_REWRITE_DIFF_TASK_DESCRIPTION = """
Rewrite the user's LinkedIn post to be more engaging.

Output BOTH versions:

ORIGINAL:
<original text>

REWRITTEN:
<rewritten text>

Rules:
- Max 120 words
- Clear point of view
- Neutral, confident tone
- Assume an informed audience
- Avoid buzzwords like "disrupt", "leverage", "game changer"
"""

CORPORATE_JARGON_TASK_DESCRIPTION = """
Convert any incoming text into a corresponding sequence of corporate jargon.
For example, if the user says, "I went to the store to buy some eggs but forgot my wallet",
you should convert it into something like:
"During a routine procurement initiative, I proceeded to the designated retail partner to acquire
essential inventory units (hen‑derived ova). However, execution was impeded when I identified
a critical absence of my primary fiscal instrument, necessitating immediate reassessment of the
transaction workflow and postponement of asset acquisition.".
"""

In [10]:
# Describe your custom task (be specific!)
# CUSTOM_TASK_DESCRIPTION = """ """

TASK_DESCRIPTION = GRAMMARLY_TASK_DESCRIPTION

# Choose the base model to train
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # Options: "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct", etc.

# Robust JSON extraction

In [11]:
def extract_json_object(text: str) -> str:
    if text is None:
        raise ValueError("No text to parse")

    t = text.strip()

    # Strip fenced code blocks if present
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", t)  # opening fence
        t = re.sub(r"\s*```$", "", t).strip()        # closing fence

    # Find first '{' or '['
    m = re.search(r"[\{\[]", t)
    if not m:
        raise ValueError(f"Could not find JSON start in: {t[:200]!r}")
    start = m.start()

    # Scan to matching closing brace/bracket while respecting strings
    stack: list[str] = []
    in_str = False
    esc = False

    for i in range(start, len(t)):
        ch = t[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue

        if ch == '"':
            in_str = True
            continue

        if ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            if not stack:
                continue
            opening = stack.pop()
            if (opening == "{" and ch != "}") or (opening == "[" and ch != "]"):
                raise ValueError("Mismatched JSON brackets")
            if not stack:
                return t[start : i + 1].strip()

    raise ValueError("Could not find end of JSON object")


# Training input generation (robust)

In [12]:
class TrainingInput(BaseModel):
    input: str = Field(description="The input text for the task")


class TrainingDataset(BaseModel):
    inputs: List[TrainingInput] = Field(description="List of training inputs")


async def generate_training_inputs(task_description: str, num_examples: int = 50) -> List[str]:
    """
    Generate diverse training inputs for the given task.
    Robust to models returning fewer items, wrong shape, code fences, or extra text.
    """
    system_prompt = f"""
You generate training inputs.

Task:
{task_description}

Return STRICT JSON only. No prose. No markdown. No code fences.

Schema:
{{
  "inputs": [
    {{"input": "string"}},
    ...
  ]
}}

Rules:
- Return exactly {num_examples} items in "inputs".
- Each "input" must be realistic and different.
- No duplicates.
""".strip()

    inputs: list[str] = []
    seen: set[str] = set()

    attempt = 0
    while attempt < 8 and len(inputs) < num_examples:
        attempt += 1
        remaining = num_examples - len(inputs)

        user_prompt = f"""
Generate {remaining} more items to complete the dataset.

Already have {len(inputs)} items.
Do not repeat any of these existing inputs:
{json.dumps(inputs, ensure_ascii=False, indent=2)}

Return STRICT JSON only with the same schema.
""".strip()

        print(f"Generating training inputs, attempt {attempt}, remaining {remaining}...")

        raw = ""
        try:
            response = await acompletion(
                model=INPUT_GENERATION_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.7,
            )

            raw = response.choices[0].message.content or ""
            if not raw.strip():
                raise ValueError("Empty model content.")

            clean = extract_json_object(raw)
            dataset = TrainingDataset.model_validate_json(clean)

            for ex in dataset.inputs:
                s = (ex.input or "").strip()
                if not s or s in seen:
                    continue
                seen.add(s)
                inputs.append(s)
                if len(inputs) >= num_examples:
                    break

        except Exception as e:
            print(f"Attempt {attempt} failed: {type(e).__name__}: {e}")
            print(f"Raw preview: {raw[:400]!r}")

    if len(inputs) < num_examples:
        raise ValueError(f"Failed to generate {num_examples} training inputs. Got {len(inputs)}.")

    return inputs

# Robust judge scoring (RULER-like) that never assumes perfect JSON

In [13]:
class JudgeItem(BaseModel):
    idx: int
    score: float
    rationale: str


class JudgeResponse(BaseModel):
    items: List[JudgeItem]


async def robust_score_group(
    group: art.TrajectoryGroup,
    judge_model: str,
    task_description: str,
    temperature: float = 0.0,
) -> art.TrajectoryGroup:
    """
    Robust scoring that assigns reward in [0, 1] per trajectory.
    Works even if the judge wraps JSON in markdown fences.
    """
    trajectories = list(group.trajectories)

    candidates = []
    for i, t in enumerate(trajectories):
        msgs = t.messages()
        assistant = msgs[-1]["content"] if msgs else ""
        candidates.append({"idx": i, "assistant_output": assistant})

    system = (
        "You are a strict evaluator.\n"
        "Return STRICT JSON only. No prose. No markdown. No code fences.\n"
        "Score each candidate output for how well it satisfies the task.\n"
        "Scores must be floats in [0, 1].\n"
    )

    user = (
        f"TASK:\n{task_description}\n\n"
        "CANDIDATES:\n"
        f"{json.dumps(candidates, ensure_ascii=False, indent=2)}\n\n"
        "Scoring rubric:\n"
        "- 1.0: Exactly matches required format and content is coherent and extracted from input\n"
        "- 0.5: Mostly matches format but missing details or minor format violations\n"
        "- 0.0: Ignores format, adds extra commentary, or does not perform task\n\n"
        'Return JSON with schema: {"items":[{"idx":0,"score":0.0,"rationale":"..."}]}\n'
    )

    resp = await acompletion(
        model=judge_model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
    )

    raw = resp.choices[0].message.content or ""
    clean = extract_json_object(raw)
    judged = JudgeResponse.model_validate_json(clean)

    score_by_idx = {}
    for it in judged.items:
        # clamp
        score_by_idx[it.idx] = max(0.0, min(1.0, float(it.score)))

    for i, t in enumerate(trajectories):
        t.reward = score_by_idx.get(i, 0.0)

    return art.TrajectoryGroup(trajectories=trajectories)

# Generate dataset

In [14]:
training_inputs = await generate_training_inputs(
    TASK_DESCRIPTION, num_examples=TRAINING_CONFIG["num_training_inputs"]
)
print(f"\nGenerated {len(training_inputs)} training inputs!")
print("\nFirst 5 examples:")
for i, input_text in enumerate(training_inputs[:5]):
    print(f"\nExample {i + 1}: {input_text}")

Generating training inputs, attempt 1, remaining 25...

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Generated 25 training inputs!

First 5 examples:

Example 1: She don't likes to eat vegetables.

Example 2: Their going to the beach next weekend.

Example 3: I have went to that museum before.

Example 4: He is more taller then his brother.

Example 5: We was planning to leave early.


In [15]:
# @title ✅ Pause before training (read + confirm)

# This cell is here on purpose:
# It prevents "Run all" from immediately creating a backend, registering a model,
# and starting a potentially expensive workflow.

print("🛑 STOP: Before you continue, make sure you want to train the model!\n")

# Set this to True ONLY when you're ready to proceed.
I_UNDERSTAND_AND_WANT_TO_CONTINUE = False

if not I_UNDERSTAND_AND_WANT_TO_CONTINUE:
    raise RuntimeError(
        "Paused intentionally. Set I_UNDERSTAND_AND_WANT_TO_CONTINUE = True, then re-run this cell."
    )

print("\n✅ Continuing to model creation and backend registration...")


🛑 STOP: Before you continue, make sure you want to train the model!



RuntimeError: Paused intentionally. Set I_UNDERSTAND_AND_WANT_TO_CONTINUE = True, then re-run this cell.

# Create model + backend

In [ ]:
random.seed(42)

model = art.TrainableModel(
    name=MODEL_NAME,
    project=PROJECT_NAME,
    base_model=BASE_MODEL,
)

# GPU friendly overrides
if torch.cuda.get_device_properties(0).major < 8:
    model._internal_config = art.dev.InternalModelConfig(
        init_args=art.dev.InitArgs(max_seq_length=MAX_SEQ_LENGTH),
        engine_args=art.dev.EngineArgs(
            enforce_eager=True,
            gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        ),
    )

backend = (
    LocalBackend(in_process=True, path="./.art")
    if torch.cuda.get_device_properties(0).major < 8
    else LocalBackend()
)

await model.register(backend)

print("Model created!")
print("Base model:", BASE_MODEL)
print("Model name:", MODEL_NAME)
print("Project name:", PROJECT_NAME)


# Weave init (optional)

In [ ]:
if os.getenv("WANDB_API_KEY", ""):
    weave.init(PROJECT_NAME, settings={"print_call_link": False})

# System prompt generation

In [ ]:
async def generate_system_prompt(task_description: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "Generate a clear, concise system prompt for a model that will perform the following task. "
                "The prompt should be direct and instructional."
            ),
        },
        {
            "role": "user",
            "content": f"Task: {task_description}\n\nGenerate a system prompt for this task.",
        },
    ]

    response = await acompletion(
        model=SYSTEM_PROMPT_GENERATION_MODEL,
        messages=messages,
        temperature=0.3,
    )

    return (response.choices[0].message.content or "").strip()


SYSTEM_PROMPT = await generate_system_prompt(TASK_DESCRIPTION)
print(f"Generated system prompt:\n\n{SYSTEM_PROMPT}")


# Rollout function

In [ ]:
class TaskInput(BaseModel):
    step: int
    input_text: str


@weave.op
async def rollout(model: art.Model, task_input: TaskInput) -> art.Trajectory:
    traj = art.Trajectory(
        reward=0.0,
        messages_and_choices=[],
        metadata={"step": task_input.step, "input": task_input.input_text},
    )

    traj.messages_and_choices = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task_input.input_text},
    ]

    litellm_model_name = f"hosted_vllm/{model.name}" if model.trainable else model.name

    response = await acompletion(
        model=litellm_model_name,
        base_url=model.inference_base_url,
        api_key=model.inference_api_key,
        temperature=0.7,
        messages=traj.messages(),
        caching=False,
    )

    traj.messages_and_choices.append(convert_litellm_choice_to_openai(response.choices[0]))
    return traj


print("\nRollout function defined!")


# Judge demo

In [ ]:
test_input = "After the last deploy the login page keeps refreshing forever on Chrome 120 on macOS. Works on Firefox. Started yesterday."
base_messages = [
    {"role": "system", "content": TASK_DESCRIPTION},
    {"role": "user", "content": test_input},
]

good_trajectory = art.Trajectory(
    messages_and_choices=[
        *base_messages,
        {
            "role": "assistant",
            "content": (
                "SUMMARY: Login page stuck in redirect loop on Chrome 120 after latest deploy\n"
                "PRIORITY: High\n"
                "STEPS TO REPRODUCE:\n"
                "1. Open Chrome 120 on macOS\n"
                "2. Navigate to /login\n"
                "3. Enter valid credentials and submit\n"
                "4. Observe repeated redirects/refresh and no successful login\n"
                "EXPECTED RESULT: User is authenticated and redirected to the dashboard\n"
                "ACTUAL RESULT: Page redirects/refreshes indefinitely and user cannot log in\n"
                "ENVIRONMENT: Chrome 120, macOS, regression after yesterday's deploy"
            ),
        },
    ],
    reward=0,
)

mediocre_trajectory = art.Trajectory(
    messages_and_choices=[
        *base_messages,
        {
            "role": "assistant",
            "content": (
                "SUMMARY: Login not working on Chrome\n"
                "PRIORITY: Medium\n"
                "STEPS TO REPRODUCE:\n"
                "1. Open Chrome\n"
                "2. Try to login\n"
                "EXPECTED RESULT: Login works\n"
                "ACTUAL RESULT: Login fails\n"
                "ENVIRONMENT: Chrome"
            ),
        },
    ],
    reward=0,
)

bad_trajectory = art.Trajectory(
    messages_and_choices=[*base_messages, {"role": "assistant", "content": "Try clearing cache."}],
    reward=0,
)

sample_group = art.TrajectoryGroup(trajectories=[good_trajectory, mediocre_trajectory, bad_trajectory])

judged_demo = await robust_score_group(
    sample_group,
    judge_model=RULER_MODEL,
    task_description=TASK_DESCRIPTION,
)

sorted_demo = sorted(judged_demo.trajectories, key=lambda t: t.reward, reverse=True)
for rank, traj in enumerate(sorted_demo, 1):
    msg = traj.messages()[-1]["content"]
    print(f"\nDemo Rank {rank}: Score {traj.reward:.3f}")
    print(f"  Response: {msg[:220]}{'...' if len(msg) > 220 else ''}")



# Training loop

In [ ]:
training_task_inputs = [TaskInput(step=0, input_text=inp) for inp in training_inputs]

training_iterator = iterate_dataset(
    training_task_inputs,
    groups_per_step=TRAINING_CONFIG["groups_per_step"],
    num_epochs=TRAINING_CONFIG["num_epochs"],
    initial_step=await model.get_step(),
)

print(f"\nStarting training with {len(training_task_inputs)} inputs...")
print(f"Training for {TRAINING_CONFIG['num_epochs']} epoch(s)")
print(f"Groups per step: {TRAINING_CONFIG['groups_per_step']}")
print(f"Rollouts per group: {TRAINING_CONFIG['rollouts_per_group']}")

for batch in training_iterator:
    print(f"\nTraining step {batch.step}, epoch {batch.epoch}, epoch step {batch.epoch_step}")
    print(f"Batch contains {len(batch.items)} inputs")

    groups = []
    for task_input in batch.items:
        task_input.step = batch.step
        groups.append(
            art.TrajectoryGroup(
                (rollout(model, task_input) for _ in range(TRAINING_CONFIG["rollouts_per_group"]))
            )
        )

    finished_groups = await art.gather_trajectory_groups(
        groups,
        pbar_desc="Generating responses",
        max_exceptions=TRAINING_CONFIG["rollouts_per_group"] * len(batch.items),
    )

    judged_groups = []
    for group in finished_groups:
        judged = None
        for _ in range(10):
            try:
                judged = await robust_score_group(
                    group,
                    judge_model=RULER_MODEL,
                    task_description=TASK_DESCRIPTION,
                )
                break
            except Exception as e:
                print(f"Error scoring group: {e}")

        if judged is None:
            raise RuntimeError("Scoring failed after retries; cannot continue training.")

        judged_groups.append(judged)

    await model.delete_checkpoints()
    await model.train(
        judged_groups,
        config=art.TrainConfig(learning_rate=TRAINING_CONFIG["learning_rate"]),
        _config={"logprob_calculation_chunk_size": 8},
    )

    print(f"Completed training step {batch.step}")

    if TRAINING_CONFIG["max_training_steps"] and batch.step >= TRAINING_CONFIG["max_training_steps"]:
        print(f"Reached maximum training steps ({TRAINING_CONFIG['max_training_steps']})")
        break

print("\n✅ Training completed!")


# Test Your Model

In [ ]:


# Generate test inputs
print("Generating test inputs...")
test_inputs = await generate_training_inputs(
    TASK_DESCRIPTION, num_examples=NUM_TEST_INPUTS
)

print(f"\n🧪 Testing the trained model on {len(test_inputs)} new inputs:\n")
print("=" * 80)

for i, test_input in enumerate(test_inputs):
    print(f"\nTest {i + 1}:")
    print(f"Input: {test_input}")

    # Run the model
    test_task_input = TaskInput(step=999, input_text=test_input)
    result_trajectory = await rollout(model, test_task_input)

    # Extract the model's response
    messages = result_trajectory.messages()
    model_response = messages[-1]["content"] if messages else "No response"

    print(f"Model output: {model_response}")
    print("-" * 80)

print("\n🎉 Testing completed!")
print(f"\nYour model '{MODEL_NAME}' has been trained to: {TASK_DESCRIPTION}")
print("\nTo use this model in production:")
print("1. The model checkpoint is saved in ./.art/")
print("2. You can load it using the vLLM library")
print(
    "3. Or continue training with more examples by adjusting the configuration at the top"
)

# Upload to Hugging Face 🤗

In [ ]:
# @title

# Adapted from Unsloth Notebooks (https://github.com/unslothai/notebooks), licensed under GNU LGPL v3.0.
# See THIRD-PARTY-NOTICES and licenses/LGPL-3.0.txt for details.

lora_model_path = (
    f".art/{model.project}/models/{model.name}/checkpoints/{await model.get_step():04d}"
)

peft_model, peft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_model_path,
    max_seq_length=16384,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)


In [ ]:
HF_ACCOUNT = "your_HF_account"
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN missing from Colab Secrets"

login(token=HF_TOKEN, add_to_git_credential=False)
print(whoami(token=HF_TOKEN))

safe_name = model.name.replace("/", "-")
repo_id = f"{HF_ACCOUNT}/{safe_name}"
create_repo(repo_id, token=HF_TOKEN, exist_ok=True)

peft_model.push_to_hub_merged(repo_id, peft_tokenizer, token=HF_TOKEN)


# Next Steps

Congrats! 🎉🚀 You've trained your own custom model using just:

Here is a rephrased version that explicitly highlights **varying and refining task descriptions** as a first-class improvement lever, while keeping the tone instructional and clean.

* A task description
* Example inputs (no outputs required)
* RULER's automatic evaluation

To further improve performance, you can iterate along several dimensions:

1. **Multiple task descriptions**
   Introduce alternative or complementary task descriptions that emphasize different aspects of “good” behavior. This helps RULER generalize across interpretations of the task rather than overfitting to a single phrasing.

2. **More diverse inputs**
   Generate a broader and more varied set of input examples to cover edge cases and realistic usage patterns.

3. **Longer training**
   Increase the number of training steps to allow the policy to stabilize and converge.

4. **More comparisons**
   Increase `rollouts_per_group` to give RULER richer comparative signals when ranking candidate behaviors.

5. **Task refinement**
   Make task descriptions more precise and explicit about priorities, constraints, and trade-offs.

Remember: RULER learns what “good” means entirely from your task descriptions and relative comparisons—no labeled outputs are required.

For more info see the [ART documentation](https://art.openpipe.ai).
